# HtsLoom data loading

Quick walkthrough of how to load streams from a session folder using the `htsloom` schema defined in [`streams.py`](streams.py).

## Setup

Import the loader and schema, then point at a session folder and pick a time window.

In [ ]:
from swc.aeon.io import load
from streams import htsloom
import pandas as pd

root  = r"C:/Data/hts/2026-04-20T133945Z/"
start = pd.Timestamp("2026-04-20T13:40:00", tz="UTC")
end   = pd.Timestamp("2026-04-20T15:00:00", tz="UTC")

## Load a few streams

Each call returns a tidy `pandas` DataFrame indexed by HARP timestamp. Streams are addressed as `htsloom.<Device>.<Stream>` — see [`streams.py`](streams.py) for the full schema.

In [ ]:
tracking_north   = load(root, htsloom.CameraNorth.HeadTail, start=start, end=end)
loom_angle = load(root, htsloom.CameraNorth.LoomAngleState, start=start, end=end)
loom_state = load(root, htsloom.CameraNorth.LoomRegionState, start=start, end=end)
video      = load(root, htsloom.CameraNest.Video, start=start, end=end)
pellets    = load(root, htsloom.Feeder1.DeliverPellet, start=start, end=end)

### Preview the head-tail tracking output

Columns are documented on the `HeadTail` class in [`streams.py`](streams.py).

In [ ]:
tracking_north.head()

## Validate every reader in the schema

Walk the `htsloom` DotMap, call `load()` on each reader, and report row counts (or the exception). A non-zero `rows` means the reader resolved files in the time window; `0` typically means no events of that kind occurred (e.g. no pellet deliveries) or the device wasn't logging.

In [ ]:
from swc.aeon.io.reader import Reader

def walk_readers(node, path=""):
    if isinstance(node, Reader):
        yield path, node
    elif hasattr(node, "items"):
        for k, v in node.items():
            yield from walk_readers(v, f"{path}.{k}" if path else k)

results = []
for path, reader in walk_readers(htsloom):
    try:
        df = load(root, reader, start=start, end=end)
        results.append((path, "ok", len(df)))
    except Exception as e:
        results.append((path, type(e).__name__, str(e)[:120]))

pd.DataFrame(results, columns=["stream", "status", "rows / error"])

## Benchmark: photodiode vs loom triggers

For each entry in `LoomTrigger/*.csv`, look at the photodiode channel for the corresponding screen and find the first sample above `PD_THRESHOLD` within `SEARCH_WINDOW_S`. The gap is the rendering + display latency.

Edit `SCREEN_TO_PD` once the physical wiring is fixed.

In [ ]:
import glob, os

SCREEN_TO_PD = {
    "ScreenNorth": "pd0",
    "ScreenEast":  "pd1",
    "ScreenSouth": "pd2",
    "ScreenWest":  "pd3",
}
PD_THRESHOLD    = 0.5
SEARCH_WINDOW_S = 0.5
HARP_EPOCH      = pd.Timestamp("1904-01-01", tz="UTC")

loom_frames = []
for f in sorted(glob.glob(os.path.join(root, "LoomTrigger", "LoomTrigger_*.csv"))):
    df = pd.read_csv(f)
    df["time"] = HARP_EPOCH + pd.to_timedelta(df["Seconds"], unit="s")
    loom_frames.append(df)
looms = (
    pd.concat(loom_frames).set_index("time").sort_index().loc[start:end]
    if loom_frames else pd.DataFrame()
)

pd_samples = load(root, htsloom.InputExpander.Photodiode, start=start, end=end)

if looms.empty or pd_samples.empty:
    print(f"looms={len(looms)}, photodiode rows={len(pd_samples)} — nothing to benchmark")
    bench = pd.DataFrame(columns=["trigger_time", "screen", "pd_channel", "latency_ms"])
else:
    rows = []
    for trig_time, screen in zip(looms.index, looms["Value.screenName"]):
        ch = SCREEN_TO_PD.get(screen)
        if ch is None:
            rows.append((trig_time, screen, None, None))
            continue
        window = pd_samples.loc[trig_time : trig_time + pd.Timedelta(seconds=SEARCH_WINDOW_S), ch]
        above = window.index[window > PD_THRESHOLD]
        latency_ms = (above[0] - trig_time).total_seconds() * 1000 if len(above) else None
        rows.append((trig_time, screen, ch, latency_ms))
    bench = pd.DataFrame(rows, columns=["trigger_time", "screen", "pd_channel", "latency_ms"])
    print(bench["latency_ms"].describe())

bench